<a href="https://colab.research.google.com/github/gitmystuff/INFO5810/blob/main/Module_01-Probability/Your_Name_Probability_Foundations_Pt_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From One Variable to Many — Modern Density Estimation

Your Name

## Getting Started

* Colab - get notebook from gitmystuff DTSC5810 repository
* Save a Copy in Drive
* Remove Copy of
* Edit your name in the cell above and the filename
* Clean up Colab Notebooks folder
* Submit shared link

## Distributions: An Introduction

Probability means more than just saying something might happen. When we roll a fair die, we don't know which number will come up — but we're far from clueless. We know there are six possible outcomes, and we know each one is equally likely. Probability is what lets us turn that uncertainty into something precise and useful.

To talk about this precisely, we need a name for "the number that comes up." We call it a random variable — a variable whose value depends on the outcome of something uncertain, like a die roll. Before you roll, the random variable doesn't have a value yet; it just has possibilities, and probability is what tells us how likely each possibility is. Now imagine rolling that die not once, but a thousand times, and keeping a running tally of how often each number comes up. You'd expect the tally for each number to settle in around the same share of the total — roughly one-sixth each. Once you list out every possible outcome of a random variable alongside how likely each one is, you've built something called a distribution. A distribution is really just an organized answer to the question "what could happen, and how likely is each option?" — and repeated observation is how we'd actually go check that answer against reality.

With two dice, you're no longer asking "what number comes up" — you're asking "what's the sum of both dice," and that changes everything. The possible outcomes now range from 2 to 12, but they're not equally likely. There's only one way to roll a 2 (1+1) or a 12 (6+6), but there are six different ways to roll a 7 (1+6, 2+5, 3+4, 4+3, 5+2, 6+1). So the distribution isn't flat anymore — it's shaped like a triangle, peaking at 7 and tapering off toward the extremes.

Notice something else about that triangle shape: the extreme outcomes — rolling a 2 or a 12 — are the least likely results, while outcomes near the middle are the most likely. That pattern, where values far from the center are rare and values near the center are common, is going to show up again and again this semester. In fact, it's the same basic logic behind hypothesis testing later on: if you observe a result so far out in the tail that it would be very unlikely to happen by chance, that's exactly what makes it "surprising" enough to pay attention to. The numbers used to draw that line — deciding how far into the tail is far enough to call something surprising — are called critical values. We're not there yet, but keep this shape in mind; you'll see it again.

## Distributions Notebook

https://github.com/gitmystuff/DSChunks/blob/main/Distributions.ipynb

This is all fine and dandy for one variable but the real world is much more complex.

## Objectives

**Where we left off:** Part 1 ended with Kernel Density Estimation (KDE) tracing the shape of a single messy variable: real datasets rarely hand you one variable in isolation. They hand you many variables at once, often correlated, often not shaped like any textbook distribution.

**This notebook is definitional and exploratory, not a mastery test.** The goal is that you leave able to say what a multivariate distribution, an exponential family, a Gaussian Mixture Model, the EM algorithm, and a copula each *are*, and that you've seen the Python tools that touch each one.

**By the end of Part 2, you should be able to:**
* Explain what a joint density is and why covariance matters when variables are considered together
* Describe, at a high level, what an exponential family is and why it matters practically
* Explain the intuition behind Gaussian Mixture Models and the EM algorithm ("guess, refine, repeat")
* Describe what a copula does that a single joint distribution formula cannot do as flexibly
* Understand how these tools connect directly to this module's graded activity

We'll carry one running example through this whole notebook: an e-commerce dataset of visitors' **time on site** and **amount spent**. It's synthetic (created below), but it's built deliberately to show multiple variables, more than one underlying subgroup, and non-normal shapes.


## 1. Recap: The Question We're Picking Up

Part 1 closed with KDE fitting a smooth curve to a single variable that turned out to be a blend of two overlapping groups. Two questions were left open:

1. What do we do when there's more than one variable — and those variables are correlated with each other?
2. What do we do when the data isn't cleanly *one* distribution, but a mixture of several?

Both questions turn out to matter enormously for real-world "knowledge discovery" work, because real datasets almost never arrive as one clean, single-variable, single-distribution sample. This notebook builds up the toolkit for both problems, in order: multivariate distributions first (many variables, still one coherent shape), then mixtures (several shapes blended together), then copulas (a different, very flexible way of handling many variables that don't share a common shape at all).


## 2. Multivariate Distributions: From a Curve to a Surface

A single-variable (univariate) distribution describes the probability or density of one variable's possible values — a curve over a line. A **multivariate distribution** describes the *joint* density of two or more variables considered together — for two variables, that's a surface over a plane, rather than a curve over a line.

The key new idea multivariate distributions introduce is **covariance**: a way of quantifying whether two variables tend to move together. If people who spend more time on a site also tend to spend more money, that's a positive covariance between "time on site" and "amount spent." A **covariance matrix** organizes this information for many variables at once: the diagonal entries are each variable's own variance, and the off-diagonal entries capture how each pair of variables moves together.

Let's build our running example: a synthetic dataset of website visitors, with two variables that we'd expect to be correlated in real life.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Simulate a single, reasonably well-behaved group of visitors for now --
# we'll deliberately complicate this dataset in Section 4.
mean_vector = [10, 20]                    # average: 10 minutes on site, $20 spent
covariance_matrix = [[9, 12],              # variance of time, covariance(time, spend)
                      [12, 25]]             # covariance(time, spend), variance of spend

visitors = rng.multivariate_normal(mean_vector, covariance_matrix, size=500)
time_on_site, amount_spent = visitors[:, 0], visitors[:, 1]

plt.figure(figsize=(6, 5))
plt.scatter(time_on_site, amount_spent, alpha=0.5, color='teal')
plt.xlabel('Time on site (minutes)')
plt.ylabel('Amount spent ($)')
plt.title('Two correlated variables: time on site vs. amount spent')
plt.show()

print("Covariance matrix:")
print(np.round(np.cov(visitors.T), 2))


Notice the upward drift in the scatter plot: visitors who spend more time on the site also tend to spend more money. That relationship is exactly what the off-diagonal entries of the covariance matrix are measuring. If those two variables were unrelated, the off-diagonal entries would sit near zero and the scatter would look like a shapeless cloud rather than a tilted ellipse.

Here's how to read that matrix, entry by entry:

- **9.92** (top-left) — the **variance of time on site**. This tells you how spread out the time-on-site values are around their mean; on its own it's not hugely interpretable (variance is in squared units), but its square root (~3.15 minutes) would give you the standard deviation — a more intuitive "typical spread."
- **23.49** (bottom-right) — the **variance of amount spent**. Same idea, just for the second variable. It's larger than 9.92, which makes sense here since spending amounts are naturally more spread out than minutes.
- **12.53** (both off-diagonal spots) — the **covariance between time on site and amount spent**. This is the number that matters most for the "do these move together" question. It's positive, which tells you the two variables increase together — more time on site tends to go with more spending. If it were near zero, the variables would be essentially unrelated; if negative, one going up would tend to come with the other going down.
- **Why the matrix is symmetric** — the covariance between time and spending is the same number whether you compute it as "time vs. spending" or "spending vs. time," so the same 12.53 appears in both off-diagonal positions. That symmetry is a general property of covariance matrices, not specific to this example.
- The diagonal tells you about each variable *alone*, and the off-diagonal tells you about the *relationship* between variables, which is exactly the new information a covariance matrix adds that a single-variable distribution can't express at all.

For the diagonal entries (variance of a single variable), the standard formula is:

$$s^2 = \text{Var}(X) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

Where:
- $x_i$ is each individual observation (each visitor's time on site, say)
- $\bar{x}$ is the sample mean
- $n$ is the number of observations
- The $n-1$ in the denominator (rather than $n$) is called **Bessel's correction** — using $n-1$ instead of $n$ gives a less biased estimate of the true population variance when you're working from a sample. This is worth flagging since you already introduced Bessel's name earlier in the notebook (the "personal equation" story) — nice chance for a callback.

Since our covariance matrix came from `np.cov()`, which computes this on real sample data, this is the version that's actually running under the hood — not the population version (which divides by $n$ instead of $n-1$).

For the off-diagonal entries (covariance between two variables), the matching formula is:

$$\text{Cov}(X, Y) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$

Notice the structural symmetry: variance is really just the special case of covariance where you compare a variable with *itself* — $\text{Cov}(X, X) = \text{Var}(X)$, which is exactly why the diagonal of a covariance matrix holds the variances.

Now let's fit a proper multivariate distribution to this data using `scipy.stats.multivariate_normal`, and visualize its joint density as a surface — the direct 2-variable extension of the single-curve PDFs from Part 1.


In [ ]:
from scipy import stats

# Fit a multivariate normal by estimating its mean vector and covariance matrix from data
fitted_mean = visitors.mean(axis=0)
fitted_cov = np.cov(visitors.T)
mvn = stats.multivariate_normal(mean=fitted_mean, cov=fitted_cov)

# Build a grid to evaluate the joint density over, and plot it as a contour map
x_grid = np.linspace(time_on_site.min() - 2, time_on_site.max() + 2, 100)
y_grid = np.linspace(amount_spent.min() - 2, amount_spent.max() + 2, 100)
X, Y = np.meshgrid(x_grid, y_grid)
positions = np.dstack((X, Y))
density = mvn.pdf(positions)

plt.figure(figsize=(6.5, 5))
plt.contourf(X, Y, density, levels=15, cmap='viridis', alpha=0.85)
plt.scatter(time_on_site, amount_spent, alpha=0.4, color='white', edgecolor='black', s=15)
plt.xlabel('Time on site (minutes)')
plt.ylabel('Amount spent ($)')
plt.title('Fitted joint density: a single multivariate normal surface')
plt.colorbar(label='Density')
plt.show()


That contour plot is the direct 2D generalization of the bell curve from Part 1: instead of one peak on a line, it's a peak on a surface, and the tilted, elongated shape of the contours reflects the positive covariance we built into the data. If time-on-site and spending were uncorrelated, those contours would be circular rather than tilted ellipses.


## 3. Exponential Families: The Same Machinery, Many Distributions

This section is conceptual — there is no need to memorize the underlying math, just the idea and why it's useful.

Across your career you'll meet a lot of named distributions: Normal, Binomial, Poisson, Gamma, Beta, Exponential. They look different, describe different kinds of data (counts, continuous measurements, proportions), and seem unrelated on the surface. It turns out that most of the distributions you'll actually use share a common underlying mathematical structure, called the **exponential family** of distributions.

Without getting into the derivation, the practical payoff is this: because these distributions share a common structure, the *same estimation machinery* — the same general approach to finding the best-fitting parameters from data (Maximum Likelihood Estimation) — works across all of them with only minor adjustments. This is precisely why a single Python function pattern (like `scipy.stats.<distribution>.fit()`) can estimate parameters for wildly different-looking distributions with the same calling convention. It's not a coincidence of software design — it reflects the shared mathematical family underneath.

Let's see that shared interface in action across three very different-looking distributions.


In [ ]:
from scipy import stats

rng = np.random.default_rng(9)

# Three differently-shaped datasets, all drawn from members of the exponential family
normal_data = rng.normal(loc=50, scale=8, size=2000)
exponential_data = rng.exponential(scale=4, size=2000)
gamma_data = rng.gamma(shape=3, scale=2, size=2000)

datasets = {
    "Normal": (normal_data, stats.norm),
    "Exponential": (exponential_data, stats.expon),
    "Gamma": (gamma_data, stats.gamma),
}

for name, (data, distribution) in datasets.items():
    fitted_params = distribution.fit(data)
    print(f"{name:12s} -- fitted parameters: {tuple(round(p, 2) for p in fitted_params)}")


Three completely different-shaped distributions (symmetric and bell-shaped, sharply right-skewed, moderately right-skewed), and the exact same `.fit()` call worked on all three. That consistency is the practical fingerprint of the exponential family: it's why libraries like `scipy.stats` can offer a single, uniform interface across dozens of distributions rather than needing a bespoke fitting algorithm for each one. You'll benefit from this every time you fit a distribution in this course without needing to know which specific optimization is happening underneath.


## 4. Gaussian Mixture Models & the EM Algorithm: When One Shape Isn't Enough

Section 2's multivariate normal worked because the data really was one coherent, elliptical cloud. Real datasets are often not so cooperative — recall the bimodal, two-lump data KDE traced in Part 1. What if, instead of one shape, your data is actually a *blend of several* shapes layered on top of each other?

A **Gaussian Mixture Model (GMM)** models exactly this situation: it assumes the data was generated by a small number of underlying groups, each shaped like its own (multivariate) normal distribution, blended together in some proportion. Unlike KDE — which makes no assumption about shape at all — a GMM assumes each hidden group is normally distributed, and tries to recover the parameters (mean, covariance, and proportion) of each group directly.

Let's deliberately complicate our visitors dataset: alongside the casual browsers from Section 2, let's add a second, distinct group — high-value buyers who spend much more time and money.


In [ ]:
rng = np.random.default_rng(21)

# Group 1: casual browsers (the same population as Section 2)
browsers = rng.multivariate_normal([10, 20], [[9, 12], [12, 25]], size=500)

# Group 2: a smaller group of high-value buyers -- a distinct center and spread
buyers = rng.multivariate_normal([35, 140], [[20, 40], [40, 400]], size=180)

all_visitors = np.vstack([browsers, buyers])

plt.figure(figsize=(6.5, 5))
plt.scatter(all_visitors[:, 0], all_visitors[:, 1], alpha=0.4, color='darkslateblue')
plt.xlabel('Time on site (minutes)')
plt.ylabel('Amount spent ($)')
plt.title('Two blended subgroups: browsers and high-value buyers')
plt.show()


If you tried to fit a single multivariate normal to this new dataset the way we did in Section 2, it would land somewhere awkwardly between the two groups, describing neither one well. A GMM instead tries to recover both groups directly.

**How does it find them, without being told upfront which points belong to which group?** This is where the **EM (Expectation-Maximization) algorithm** comes in, and it's worth naming the callback to Part 1 explicitly: this is the same "guess, measure the fit, refine" instinct behind Gauss's least squares. EM alternates between two steps, repeated until the fit stops improving:

* **E-step (Expectation):** given the *current* guessed parameters for each group, estimate the probability that each individual data point belongs to each group.
* **M-step (Maximization):** given those probability-weighted group assignments, re-estimate each group's parameters (mean, covariance, proportion) to best fit the points now assigned to it.

Guess → check the fit → refine the guess → repeat. Let's run it and see whether it recovers the two groups we built in.


In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=2, random_state=0)
gmm.fit(all_visitors)

cluster_assignment = gmm.predict(all_visitors)

print("Discovered group means (time on site, amount spent):")
print(np.round(gmm.means_, 1))
print()
print("Discovered group proportions (weights):", np.round(gmm.weights_, 2))

plt.figure(figsize=(6.5, 5))
plt.scatter(all_visitors[:, 0], all_visitors[:, 1], c=cluster_assignment,
            cmap='coolwarm', alpha=0.5)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='gold', marker='X', s=250,
            edgecolor='black', linewidth=1.5, label='Discovered group centers')
plt.xlabel('Time on site (minutes)')
plt.ylabel('Amount spent ($)')
plt.title('GMM recovers the two underlying subgroups')
plt.legend()
plt.show()

The two discovered group centers should land close to the `[10, 20]` browsers and `[35, 140]` buyers we simulated — the algorithm never saw the true group labels, only the raw scatter of points, and recovered the underlying structure anyway. This is the core idea behind **unsupervised clustering**: no one told the model the "right answer" for each point; it inferred structure from the data's shape alone.

Notice something else worth naming explicitly: GMM is a *parametric* alternative to KDE. KDE (Part 1) makes no assumption about shape and just smooths the data directly — flexible, but harder to summarize compactly. GMM assumes the data is made of a specific number of normal-shaped groups — less flexible, but the result is a compact, interpretable set of parameters (a handful of means, covariances, and weights) rather than a smoothed curve over every point.


## 5. Copulas: Separating "Shape" from "Relationship"

Section 2's multivariate normal had a hidden assumption worth surfacing: it assumed *both* variables were individually normally distributed, and that their relationship followed the specific correlation structure built into the normal distribution's math. Real variables are often not each individually normal — spending amounts are typically right-skewed (most people spend a little, a few spend a lot), for instance — even when they're still clearly correlated with something else.

**Copulas** solve this by separating two questions that a single joint-distribution formula bundles together:

1. What does *each individual variable's* distribution look like on its own (its **marginal distribution**)?
2. How do the variables **relate** to each other, independent of what each one's individual shape is?

This separation is formalized by a result called **Sklar's Theorem**, which says (without getting into the proof) that any joint distribution can be decomposed into its marginal distributions plus a copula function capturing the dependence structure between them. The practical benefit: you can mix and match — model "amount spent" as right-skewed, model "time on site" as its own separate shape, and separately specify how strongly they move together, rather than being locked into one formula that forces both.

Let's build a dataset that makes this concrete: two variables with realistically skewed (non-normal) individual shapes, that are still meaningfully correlated with each other — and then fit a copula to recover just the dependence structure.


In [ ]:
from statsmodels.distributions.copula.api import CopulaDistribution, GaussianCopula

rng = np.random.default_rng(3)

# Define each variable's own individual shape (its marginal distribution) --
# both realistically skewed, not normal.
time_marginal = stats.gamma(a=2.2, scale=6)          # right-skewed time-on-site
spend_marginal = stats.lognorm(s=0.6, scale=np.exp(3.2))  # heavily right-skewed spending

# Define how strongly the two variables should move together, independent of their shapes
true_correlation = [[1.0, 0.75],
                     [0.75, 1.0]]
dependence_structure = GaussianCopula(corr=true_correlation, k_dim=2)

# Combine shape + dependence into one joint distribution, and sample from it
joint_distribution = CopulaDistribution(dependence_structure, [time_marginal, spend_marginal])
simulated_visitors = joint_distribution.rvs(1000, random_state=rng)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].hist(simulated_visitors[:, 0], bins=40, color='seagreen')
axes[0].set_title('Marginal: time on site\n(right-skewed, not normal)')

axes[1].hist(simulated_visitors[:, 1], bins=40, color='indianred')
axes[1].set_title('Marginal: amount spent\n(right-skewed, not normal)')

axes[2].scatter(simulated_visitors[:, 0], simulated_visitors[:, 1], alpha=0.4, color='slateblue')
axes[2].set_title('Joint relationship\n(still clearly correlated)')
axes[2].set_xlabel('Time on site')
axes[2].set_ylabel('Amount spent')

plt.tight_layout()
plt.show()


Look at the two histograms on the left: neither variable is remotely bell-shaped, yet the scatter plot on the right still shows a clear, strong relationship between them. A single multivariate normal, like the one we used in Section 2, could not represent this data well, because it forces both marginal shapes to be normal. A copula has no such restriction — it let us keep each variable's realistic, skewed shape while still modeling how strongly they relate.

Now let's do the reverse: given only the data (as if we'd collected it from real visitors, without knowing the "true" correlation we used to generate it), can we recover the strength of the dependence between the two variables?


In [ ]:
# Fit a Gaussian copula's dependence parameter directly from the simulated data
fitted_copula = GaussianCopula(k_dim=2)
recovered_correlation = fitted_copula.fit_corr_param(simulated_visitors)

print(f"True correlation used to generate the data:      0.75")
print(f"Correlation recovered by fitting the copula:      {recovered_correlation:.3f}")


The fitted copula recovers the dependence strength closely, using only the raw data — without ever needing to know (or force) what shape each individual variable was supposed to follow. This is exactly the workflow you'll use in **Activity 1**: fitting parametric copulas to complex tabular data where individual columns rarely look like clean, familiar, single-named distributions, but the *relationships between columns* are still exactly what you're often most interested in.


## 6. Closing: Back to the Thesis

Part 1 argued that probability is the discipline of formally reasoning about uncertainty — and that statistics, and by extension knowledge discovery, depend on it entirely. Part 2 has been that same argument applied to harder, more realistic data:

* **Multivariate distributions** extended "uncertainty about one variable" to "uncertainty about several variables at once, including how they relate."
* **Exponential families** showed that the tools for handling that uncertainty share a common mathematical backbone, which is why the same code patterns keep working across very different-looking distributions.
* **GMM and EM** brought back Gauss's "guess, measure the error, refine" instinct from Part 1 — applied not to a single noisy measurement, but to discovering hidden subgroups inside a blended dataset.
* **Copulas** pushed the flexibility further still, separating what each variable looks like from how variables relate, so that realistic, messy, non-normal data can still be modeled honestly.

Every one of these tools exists for the same underlying reason the whole discipline exists: real data is uncertain, messy, and multidimensional, and the goal of knowledge discovery is to make responsible, quantified claims about it anyway — not to pretend the uncertainty isn't there.


## Summary

**We started with the idea that humans are bad at randomness.** A man won the lottery and explained it by saying he dreamed of 7 for seven nights and "7 times 7 is 48" (it's 49). It's a funny story, but it points at something real: people build private theories out of coincidence, and probability exists as the formal corrective — a way to reason about uncertainty carefully instead of trusting a gut feeling.

**Then we got precise about what "probability" even means.** Rolling a die, you don't know exactly what number will come up — but you're not clueless either. You know there are six possible outcomes and that each is equally likely. Probability is what turns that vague sense of uncertainty into something exact.

**To talk about this more formally, we gave "the number that comes up" a name: a random variable.** Before you roll, it doesn't have a value yet — just possibilities. If you rolled the die a thousand times and kept a running tally, each number would settle in around one-sixth of the rolls. That's the idea of a **distribution**: an organized answer to "what could happen, and how likely is each option?"

**Then we asked: does that change with two dice?** It does, and in an interesting way. One die gives a flat distribution — every outcome equally likely. Two dice summed together gives a triangle shape, peaking at 7, because there's only one way to make a 2 (1+1) but six ways to make a 7. Extreme outcomes (2 or 12) are rare; middle outcomes are common. That specific "rare in the tails, common in the middle" pattern is a preview of two big ideas that come later: **critical values** in hypothesis testing (deciding how far into the tail counts as "surprising"), and eventually the **Central Limit Theorem** — the idea that summing enough independent random things tends toward a bell curve, no matter what each individual thing looked like.

**Next we moved from one variable to two at once — time on a website and money spent.** When you have two variables, a new question becomes possible: do they move together? That's **covariance** — a positive number means "when one goes up, the other tends to go up too." A **covariance matrix** is just a compact way of storing three numbers at once: how spread out variable one is (variance), how spread out variable two is (variance), and how the two relate (covariance). The diagonal is each variable alone; the off-diagonal is the relationship between them — genuinely new information that no single-variable distribution could ever capture.

**Finally, we looked at what happens when your data isn't one clean group but two blended together** — casual browsers and high-value buyers, mixed into the same scatterplot. A Gaussian Mixture Model tries to discover those hidden groups automatically, without ever being told which point belongs to which group — it just looks at the shape of the data and infers the structure. That's the essence of **unsupervised clustering**.

**The thread tying all of it together:** probability isn't just "the chance something happens." It's the formal toolkit for describing what's possible, how likely each possibility is, how multiple uncertain things relate to each other, and how to responsibly separate "real pattern" from "coincidence" — which is exactly what the lottery-guy story got wrong at the very start.